In [2]:
import pandas as pd 
df = pd.read_csv("conversation_bundle_flat.csv")
df_with_3_annotations = df[df["n_annotations"] == 3]
df_with_3_annotations.head()

,conversation_id,n_annotations,metadata,prod_log,outcome,an1,an2,an3
2,00ab8e48-5dd9-3bb5-c054-216ab58c9fda,3,"{'language': 'hindi', 'zone': 'north', 'dpd': ...",{'conversation_id': '00ab8e48-5dd9-3bb5-c054-2...,{'conversation_id': '00ab8e48-5dd9-3bb5-c054-2...,{'conversation_id': '00ab8e48-5dd9-3bb5-c054-2...,{'conversation_id': '00ab8e48-5dd9-3bb5-c054-2...,{'conversation_id': '00ab8e48-5dd9-3bb5-c054-2...
5,0116e0f9-0373-5d3a-149d-7b18d3c3c579,3,"{'language': 'hindi', 'zone': 'north', 'dpd': ...",{'conversation_id': '0116e0f9-0373-5d3a-149d-7...,{'conversation_id': '0116e0f9-0373-5d3a-149d-7...,{'conversation_id': '0116e0f9-0373-5d3a-149d-7...,{'conversation_id': '0116e0f9-0373-5d3a-149d-7...,{'conversation_id': '0116e0f9-0373-5d3a-149d-7...
7,01d2bc92-a9d8-1c8c-3a16-3bc390f1de16,3,"{'language': 'hinglish', 'zone': 'east', 'dpd'...",{'conversation_id': '01d2bc92-a9d8-1c8c-3a16-3...,{'conversation_id': '01d2bc92-a9d8-1c8c-3a16-3...,{'conversation_id': '01d2bc92-a9d8-1c8c-3a16-3...,{'conversation_id': '01d2bc92-a9d8-1c8c-3a16-3...,{'conversation_id': '01d2bc92-a9d8-1c8c-3a16-3...
10,0261f428-dcd8-21c5-227c-f8d0502cc8d2,3,"{'language': 'english', 'zone': 'north', 'dpd'...",{'conversation_id': '0261f428-dcd8-21c5-227c-f...,{'conversation_id': '0261f428-dcd8-21c5-227c-f...,{'conversation_id': '0261f428-dcd8-21c5-227c-f...,{'conversation_id': '0261f428-dcd8-21c5-227c-f...,{'conversation_id': '0261f428-dcd8-21c5-227c-f...
11,032101dc-a78c-9aa9-c8e2-5f92b42f1dfa,3,"{'language': 'english', 'zone': 'north', 'dpd'...",{'conversation_id': '032101dc-a78c-9aa9-c8e2-5...,{'conversation_id': '032101dc-a78c-9aa9-c8e2-5...,{'conversation_id': '032101dc-a78c-9aa9-c8e2-5...,{'conversation_id': '032101dc-a78c-9aa9-c8e2-5...,{'conversation_id': '032101dc-a78c-9aa9-c8e2-5...


In [13]:
# Get the annotation for the first row with 3 annotations and print the keys for an1, an2, an3
import ast

first_row = df_with_3_annotations.iloc[0]
for annotator in ['an1', 'an2', 'an3']:
    entry = first_row[annotator]
    if isinstance(entry, str):
        entry_dict = ast.literal_eval(entry)
    else:
        entry_dict = entry
    print(f"Keys for {annotator}:", list(entry_dict.keys()))

Keys for an1: ['conversation_id', 'quality_score', 'failure_points', 'risk_flags', 'overall_assessment', '_annotator', '_model']
Keys for an2: ['conversation_id', 'quality_score', 'failure_points', 'risk_flags', 'overall_assessment', '_annotator', '_model']
Keys for an3: ['conversation_id', 'quality_score', 'failure_points', 'risk_flags', 'overall_assessment', '_annotator', '_model']


In [12]:
# For each row in the dataframe with 3 annotations, print the values of an1, an2, an3

import ast

for idx, row in df_with_3_annotations.iterrows():
    print(f"conversation_id: {row['conversation_id']}")
    # Parse risk_flags for each annotator and sort
    risk_flags_dict = {}
    for annotator in ['an1', 'an2', 'an3']:
        entry = row[annotator]
        if isinstance(entry, str):
            entry_dict = ast.literal_eval(entry)
        else:
            entry_dict = entry
        flags = entry_dict.get("risk_flags", [])
        risk_flags_dict[annotator] = set(flags)
        print(f"  {annotator} risk_flags (sorted):", sorted(flags))
    # Calculate union, common (intersection), pairwise intersections, and unique-to-one
    all_flags = list(risk_flags_dict.values())
    union = set.union(*all_flags)
    intersection = set.intersection(*all_flags)
    print("  >> UNION of all risk_flags     :", sorted(union))
    print("  >> COMMON to all annotators    :", sorted(intersection) if intersection else "-")
    # Flags present in exactly two annotators
    pairwise = []
    annotators = ['an1', 'an2', 'an3']
    for i in range(3):
        j, k = (i+1)%3, (i+2)%3
        pair = risk_flags_dict[annotators[j]] & risk_flags_dict[annotators[k]] - intersection
        if pair:
            pairwise.append((f"{annotators[j]}+{annotators[k]}", sorted(pair)))
    # Remove duplicates in pairwise (since same flag may appear multiple times)
    seen_pw = set()
    for combo, pflags in pairwise:
        filtered = [x for x in pflags if x not in seen_pw]
        if filtered:
            print(f"  >> In exactly 2 annotators ({combo}):", filtered)
            seen_pw.update(filtered)
    # Flags present in only one annotator
    for annotator in annotators:
        only_this = risk_flags_dict[annotator] - (risk_flags_dict[annotators[(annotators.index(annotator)+1)%3]] | risk_flags_dict[annotators[(annotators.index(annotator)+2)%3]])
        if only_this:
            print(f"  >> ONLY in {annotator}:", sorted(only_this))
    print()

conversation_id: 00ab8e48-5dd9-3bb5-c054-216ab58c9fda
  an1 risk_flags (sorted): ['compliance_concern', 'escalation_missed']
  an2 risk_flags (sorted): ['compliance_concern']
  an3 risk_flags (sorted): ['escalation_missed']
  >> UNION of all risk_flags     : ['compliance_concern', 'escalation_missed']
  >> COMMON to all annotators    : -
  >> In exactly 2 annotators (an3+an1): ['escalation_missed']
  >> In exactly 2 annotators (an1+an2): ['compliance_concern']

conversation_id: 0116e0f9-0373-5d3a-149d-7b18d3c3c579
  an1 risk_flags (sorted): ['compliance_concern', 'escalation_missed']
  an2 risk_flags (sorted): ['compliance_concern', 'hardship_ignored']
  an3 risk_flags (sorted): []
  >> UNION of all risk_flags     : ['compliance_concern', 'escalation_missed', 'hardship_ignored']
  >> COMMON to all annotators    : -
  >> In exactly 2 annotators (an1+an2): ['compliance_concern']
  >> ONLY in an1: ['escalation_missed']
  >> ONLY in an2: ['hardship_ignored']

conversation_id: 01d2bc92-a9d8